# Emergency canonical ZINC/QM9 worker

This one notebook controls the fixed 30-checkpoint corpus. Put `zinc_qm9_best_checkpoints.tar` and `zinc_qm9_best_checkpoints.tar.sha256` directly in `DRIVE_FOLDER`. Run `MODE=setup` once: it verifies/extracts the corpus, warms the shared ZINC and QM9 datasets under an exclusive Drive lock, and creates both 30 commit-pinned single-checkpoint notebooks under `worker_notebooks/` and four commit-pinned sequential notebooks under `queue_notebooks/`. Rerunning setup after an orchestration update rehashes and reuses intact staged assets, then refreshes the pinned notebooks. For four Colab GPUs, open `queue_01_of_04.ipynb` through `queue_04_of_04.ipynb` and **Run all** in each. Together they cover all 30 checkpoints in disjoint 8/8/7/7 queues.

`queue` runs every index in `WORKER_INDICES` sequentially in one runtime. A rerun revalidates and skips completed checkpoints, then resumes the first missing or incomplete checkpoint; score and carriage payloads are released between components and checkpoints. `preflight` verifies one selection without model work. `smoke` runs the selected checkpoint against a separate three-graph cache. `status` prints the 30-worker matrix. Run `finalize` after all rows show `complete=True`; finalization reopens every artifact and performs the authoritative full postflight. Production starts with all 48 graphs on an A100-80GB and halves a failing CUDA batch automatically. Do not launch the same worker or queue twice concurrently.


In [ ]:
# Controls: generated copies pin MODE, their worker selector(s), and the revision.
MODE = "setup"  # @param ["setup", "preflight", "smoke", "worker", "queue", "status", "finalize"]
WORKER_INDEX = 0  # @param {type:"integer"}
WORKER_INDICES = ""  # @param {type:"string"}
DRIVE_FOLDER = "/content/drive/MyDrive/graph_specialisation_metrics/multi_seed_models"

# Optional explicit selector: set both to override WORKER_INDEX.
TASK = ""
TRAIN_SEED = -1
# Zero selects the GPU-aware profile: A100/H100 >=75 GiB -> all 48 graphs.
GRAPHS_PER_BATCH = 0
ACCELERATOR = "cuda:0"
STRICT_AUDITS = False
# Single-worker stale-lock override. Queue mode instead names one exact index below.
RECLAIM_STALE_LOCK = False
RECLAIM_WORKER_INDEX = -1  # @param {type:"integer"}

REPO_URL = "https://github.com/joshgreenwa/Graph-Specialisation-and-Metrics.git"
REPO_REVISION = "expansion/carriage_experiments"
REPO_DIR = "/content/Graph-Specialisation-and-Metrics"
GITHUB_SECRET = "dissertation_key"


In [ ]:
# Mount Drive and check out the requested branch/commit without exposing the token.
import importlib
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
token = userdata.get(GITHUB_SECRET) or os.environ.get(GITHUB_SECRET)
suffix = REPO_URL.removeprefix("https://github.com/")
clone_url = (
    f"https://x-access-token:{quote(str(token).strip(), safe='')}@github.com/{suffix}"
    if token else REPO_URL
)
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", clone_url, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", clone_url], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_REVISION], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", REPO_URL], check=True)
for entry in (str(repo), str(repo / "src")):
    if entry in sys.path:
        sys.path.remove(entry)
    sys.path.insert(0, entry)
importlib.invalidate_caches()
controller_module = "experiments.methodology.zinc_qm9_canonical_colab_worker"
for name in tuple(sys.modules):
    if (
        name == controller_module
        or name == "graph_specialisation_metrics"
        or name.startswith("graph_specialisation_metrics.")
    ):
        del sys.modules[name]
methodology_package = sys.modules.get("experiments.methodology")
if methodology_package is not None:
    vars(methodology_package).pop("zinc_qm9_canonical_colab_worker", None)


In [ ]:
# Execute setup, one isolated worker, a sequential queue, smoke/preflight, status, or finalization.
from experiments.methodology.zinc_qm9_canonical_colab_worker import run_frontend

result = run_frontend(
    mode=MODE,
    drive_folder=DRIVE_FOLDER,
    worker_index=WORKER_INDEX,
    worker_indices=WORKER_INDICES,
    task=(TASK or None),
    seed=(TRAIN_SEED if TASK else None),
    graphs_per_batch=(GRAPHS_PER_BATCH or None),
    accelerator=ACCELERATOR,
    strict_audits=STRICT_AUDITS,
    reclaim_stale_lock=RECLAIM_STALE_LOCK,
    reclaim_worker_index=RECLAIM_WORKER_INDEX,
)
result
